<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_turbo4%2B%E5%AF%BC%E6%BC%94%E5%8F%B0%E7%BB%BC%E5%90%88%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3 Turbo + 导演台（Colab A100）
按 Cell 1→4 运行。工作流来自随附 JSON；旧素材引用已清空，首尾帧请直接在 ComfyUI 的 `MiniMaxH3Director` 时间线上传。ComfyUI、Director、KJNodes、Turbo 节点和 Manager 每次都会同步最新提交。

In [ ]:
# Cell 1：A100 检查与配置
import os,sys,re,json,gzip,base64,hashlib,shutil,subprocess,time,urllib.request
from pathlib import Path
ROOT=Path('/content'); COMFY=ROOT/'ComfyUI'; PORT=8188
WORKFLOW='MiniMaxH3_导演台全能工作流_A100'
HF=ROOT/'hf_cache'; HF.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(HF); os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
def sh(cmd,cwd=None,check=True):
 print('+',cmd); return subprocess.run(cmd,shell=True,cwd=str(cwd) if cwd else None,text=True,check=check,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
q=sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits').stdout.strip().split(',')
name=q[0].strip(); vram=float(q[1])/1024
print(f'GPU={name}, VRAM={vram:.1f}GB, free disk={shutil.disk_usage(ROOT).free/1024**3:.1f}GB')
if 'A100' not in name.upper() or vram<38: raise RuntimeError('请选择 Google Colab A100 40GB + 高 RAM 运行时')
try:
 from google.colab import userdata
 token=userdata.get('HF_TOKEN')
 if token: os.environ['HF_TOKEN']=token
except Exception: pass
base='https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/c312e23180176fe18b62218c68075d6498e86a56/example_workflows/minimax_h3_director_a100.b64.'
expected=['c33b3e20732ff08a7d8766d2affff4d89e999f960c7a62f1f3ac8ef7d3208711','56d0bd9c10e1ad7ebacc6ae06d197ce72a95f466c64961592f6353c21dc7c26c','bd91f6d227f21e6fe2939160c99e9553039efea977feed273ad89796d6be8da6']
parts=[]
for i,want in enumerate(expected,1):
 s=''.join(urllib.request.urlopen(base+str(i)).read().decode('ascii').split())
 got=hashlib.sha256(s.encode()).hexdigest()
 if got!=want: raise RuntimeError(f'工作流分片 {i} 校验失败：{got}')
 parts.append(s)
s=''.join(parts); s += '=' * (-len(s) % 4)
raw=gzip.decompress(base64.b64decode(s,validate=True))
json.loads(raw)
(ROOT/'workflow.json').write_bytes(raw)
print('工作流已释放；素材将在 ComfyUI 内上传。')

In [ ]:
# Cell 2：同步最新 ComfyUI 与节点
def sync(url,path,branch=None):
 path=Path(path)
 if (path/'.git').exists():
  sh(f"git remote set-url origin '{url}'",path); sh(f"git fetch --depth 1 origin {branch or 'HEAD'}",path); sh('git reset --hard FETCH_HEAD && git clean -fd',path)
 else:
  shutil.rmtree(path,ignore_errors=True); sh(f"git clone --depth 1 {'--branch '+branch if branch else ''} '{url}' '{path}'")
 print(path.name,sh("git log -1 --format='%h %ad %s' --date=short",path).stdout.strip()); return path
def requirements(path):
 f=Path(path)/'requirements.txt'
 if not f.exists(): return
 lines=[x for x in f.read_text(errors='ignore').splitlines() if x.strip() and not re.match(r'^(torch|torchvision|torchaudio|triton)([<=>~!\[]|$)',x.strip(),re.I)]
 t=ROOT/'requirements.safe.txt'; t.write_text('\n'.join(lines)); sh(f"{sys.executable} -m pip install -q -r '{t}'")
sync('https://github.com/Comfy-Org/ComfyUI.git',COMFY,'master'); requirements(COMFY)
sh(f"{sys.executable} -m pip install -q 'huggingface_hub[hf_xet]' requests")
cn=COMFY/'custom_nodes'; cn.mkdir(parents=True,exist_ok=True)
repos=[('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director'),('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes'),('https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git','ComfyUI-MiniMax-H3-Turbo'),('https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Manager')]
for u,n in repos: requirements(sync(u,cn/n))
print('最新 ComfyUI/Director/自定义节点已就绪。')

In [ ]:
# Cell 3：下载 A100 模型
from huggingface_hub import hf_hub_download
def model(repo,file,sub):
 src=Path(hf_hub_download(repo_id=repo,filename=file,cache_dir=str(HF),token=os.environ.get('HF_TOKEN'))).resolve(); d=COMFY/'models'/sub; d.mkdir(parents=True,exist_ok=True); dst=d/Path(file).name
 if dst.exists() or dst.is_symlink(): dst.unlink()
 try: os.link(src,dst)
 except OSError: os.symlink(src,dst)
 print('✓',dst.name)
r='Comfy-Org/MiniMax-H3'
for f,d in [('diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors','diffusion_models'),('text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors','text_encoders'),('vae/minimax_h3_video_vae_fp16.safetensors','vae'),('vae/minimax_h3_audio_vae_fp32.safetensors','vae')]: model(r,f,d)
model('larryvrh/MiniMax-H3-Turbo-Lora','minimax_h3_turbo_v4_step600_ema.safetensors','loras')
u=COMFY/'models'/'upscale_models'; u.mkdir(parents=True,exist_ok=True); p=u/'4x-UltraSharp.pth'
if not p.exists(): urllib.request.urlretrieve('https://huggingface.co/embed/upscale/resolve/main/4x-UltraSharp.pth',p)
wdir=COMFY/'user'/'default'/'workflows'; wdir.mkdir(parents=True,exist_ok=True); shutil.copy2(ROOT/'workflow.json',wdir/f'{WORKFLOW}.json')
print('模型与工作流已就绪。')

In [ ]:
# Cell 4：启动 ComfyUI，并通过 Cloudflare 临时隧道访问
sh("pkill -f '/content/ComfyUI/main.py' || true",check=False)
sh("pkill -f '/content/cloudflared' || true",check=False)
log=open(ROOT/'comfy.log','w')
proc=subprocess.Popen([sys.executable,'main.py','--listen','0.0.0.0','--port',str(PORT),'--preview-method','auto','--disable-auto-launch'],cwd=COMFY,stdout=log,stderr=subprocess.STDOUT)
for _ in range(180):
 try:
  if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats',timeout=2).status==200: break
 except Exception: time.sleep(2)
else: raise RuntimeError('ComfyUI 启动失败，请查看 /content/comfy.log')
CF=ROOT/'cloudflared'
if not CF.exists() or CF.stat().st_size<10_000_000:
 sh(f"wget -q --tries=3 -O '{CF}' 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'")
 CF.chmod(0o755)
cf_log=ROOT/'cloudflared.log'; cf_log.write_text('')
cf_handle=open(cf_log,'w')
tunnel=subprocess.Popen([str(CF),'tunnel','--no-autoupdate','--protocol','http2','--url',f'http://127.0.0.1:{PORT}'],stdout=cf_handle,stderr=subprocess.STDOUT)
public_url=None; text=''
for _ in range(90):
 time.sleep(1); text=cf_log.read_text(errors='ignore')
 m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
 if m: public_url=m.group(0); break
 if tunnel.poll() is not None: break
if not public_url:
 print(text[-3000:]); raise RuntimeError('Cloudflare 隧道创建失败，请重新运行 Cell 4')
print('READY：Workflows →',WORKFLOW)
print('临时链接（不要分享，运行时结束后自动失效）：',public_url)
from IPython.display import HTML,display
display(HTML(f'<a href="{public_url}" target="_blank" rel="noopener" style="display:inline-block;padding:12px 20px;background:#238636;color:white;border-radius:8px;text-decoration:none;font-weight:700">打开 ComfyUI</a>'))
